# 40 - Reto 1: Chatbot Design (Capa de Orquestacion Conversacional)

## Proposito
Definir la capa conversacional/orquestadora del sistema analitico de Reto 1. Este notebook no rehace calculos de NB20 o NB30: diseña como el chatbot interpreta preguntas, decide herramientas, aplica reglas y responde con evidencia y limites.

## Relacion con NB20 y NB30
- NB20 aporta contrato semantico, intents, comparabilidad y reglas de lenguaje.
- NB30 aporta candidatos de insight, scoring, caveats y narrativa templada.
- NB40 define como el chatbot consume esas capas sin inventar calculos.

## Outputs esperados
- docs/architecture/reto1_chatbot_contract.md
- reports/reto1/chatbot_design_summary.md
- reports/reto1/golden_conversation_flows.md
- reports/reto1/chatbot_planner_schema.json (opcional util para implementacion)

## 0) Setup y contexto

### Que se hace
Se cargan contratos semanticos, reglas de negocio, intents y outputs del insight engine para diseñar la capa conversacional gobernada.

### Por que se hace
El diseño del chatbot debe operar sobre artefactos auditables del sistema analitico, no sobre inferencias libres del LLM.

In [14]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import textwrap

import pandas as pd
import yaml

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / 'config').exists() and (p / 'reports' / 'reto1').exists():
            return p
    raise FileNotFoundError('No se encontro root del proyecto.')

ROOT = find_project_root()
CONFIG_DIR = ROOT / 'config'
REPORTS_DIR = ROOT / 'reports' / 'reto1'
ARCH_DIR = ROOT / 'docs' / 'architecture'
ARCH_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_DIR / 'metrics.yaml', 'r', encoding='utf-8') as f:
    metrics_cfg = yaml.safe_load(f)
with open(CONFIG_DIR / 'business_rules.yaml', 'r', encoding='utf-8') as f:
    business_rules = yaml.safe_load(f)
with open(CONFIG_DIR / 'question_types.yaml', 'r', encoding='utf-8') as f:
    question_types_cfg = yaml.safe_load(f)

semantic_report = json.loads((REPORTS_DIR / 'semantic_layer_report.json').read_text(encoding='utf-8')) if (REPORTS_DIR / 'semantic_layer_report.json').exists() else {}
insight_report = json.loads((REPORTS_DIR / 'insight_engine_report.json').read_text(encoding='utf-8')) if (REPORTS_DIR / 'insight_engine_report.json').exists() else {}
insight_candidates = pd.read_parquet(REPORTS_DIR / 'insight_candidates.parquet') if (REPORTS_DIR / 'insight_candidates.parquet').exists() else pd.DataFrame()

metrics_catalog = pd.DataFrame(metrics_cfg.get('metrics', []))
intents_catalog = pd.DataFrame(question_types_cfg.get('question_types', []))

print('ROOT:', ROOT)
print('intents:', intents_catalog.shape[0])
print('metrics:', metrics_catalog.shape[0])
print('insight_candidates:', insight_candidates.shape)

ROOT: /home/thechieft/projects/IAeng
intents: 8
metrics: 13
insight_candidates: (12443, 35)


## 1) Arquitectura conversacional propuesta

### Que hace el chatbot
- Interpreta intencion del usuario y extrae parametros.
- Orquesta llamadas a funciones deterministicas.
- Ensambla respuesta en formato estable con evidencia y caveats.
- Gestiona contexto conversacional minimo para follow-ups.

### Que NO hace el chatbot
- No calcula metricas sobre datos crudos.
- No reemplaza semantic layer ni insight engine.
- No afirma causalidad con evidencia asociativa.
- No ignora reglas de comparabilidad o metricas suspendidas.

### Papel del LLM
El LLM se usa como planner/orquestador y sintetizador controlado. El calculo permanece fuera del LLM en herramientas deterministicas.

### Diagrama de flujo conversacional

```
Usuario
  │  pregunta en lenguaje natural
  ▼
┌─────────────────────────────────────────────────────┐
│  LLM — PLANNER / ORQUESTADOR (no calcula datos)     │
│                                                     │
│  1. Clasifica intent                                │
│  2. Extrae parámetros (metric, entity, time, ...)   │
│  3. Valida contra semantic contract (NB20)          │
│  4. Decide: ejecutar | pedir aclaración | rechazar  │
│  5. Construye ChatbotExecutionPlan                  │
└─────────────────┬───────────────────────────────────┘
                  │  plan validado
        ┌─────────┴──────────┐
        │                    │
        ▼                    ▼
┌───────────────┐   ┌────────────────────┐
│  DETERMINISTIC│   │  INSIGHT ENGINE    │
│  TOOL LAYER   │   │  (NB30 outputs)    │
│               │   │                   │
│ get_metric_v  │   │ insight_candidates │
│ aggregate_m   │   │ insight_engine_r.. │
│ rank_by_m     │   │                   │
│ get_trend     │   │ Lectura de parquet │
│ compare_seg   │   │ o re-ejecución     │
│ screen_by_c   │   └────────────────────┘
└───────┬───────┘            │
        │                    │
        └─────────┬──────────┘
                  │  resultados determinísticos
                  ▼
┌─────────────────────────────────────────────────────┐
│  LLM — RENDERER CONTROLADO                          │
│                                                     │
│  - Aplica response_contract                         │
│  - Aplica language_rules (NB20)                     │
│  - Agrega caveats según validation_status           │
│  - Formatea evidencia + visualización               │
│  - Sugiere follow-up questions                      │
└─────────────────────────────────────────────────────┘
  │
  ▼
Respuesta estructurada al usuario
```

**Principio central:** el LLM toca el texto, nunca los números.  
Los números los producen funciones determinísticas auditables.  
El LLM elige qué función llamar y cómo presentar el resultado — no cómo calcularlo.

## 2) Catalogo de intents operativos

Mapeo de intents semanticos (NB20) a comportamiento conversacional y de orquestacion.

In [2]:
def list_to_text(x):
    if isinstance(x, list):
        return ', '.join(str(v) for v in x)
    return str(x) if x is not None else ''

intents_ops = intents_catalog.copy()
intents_ops['required_params_text'] = intents_ops.get('required_params', pd.Series([[]] * len(intents_ops))).apply(list_to_text)
intents_ops['optional_params_text'] = intents_ops.get('optional_params', pd.Series([[]] * len(intents_ops))).apply(list_to_text)
intents_ops['future_function_name'] = intents_ops.get('future_function', '')
intents_ops['requires_visualization'] = intents_ops.get('default_visualization', '').astype(str).ne('')
intents_ops['direct_answer_possible'] = intents_ops['id'].isin(['query'])
intents_ops['requires_clarification_by_default'] = intents_ops['id'].isin(['multivariable_filter', 'hypothesis_request'])
intents_ops['requires_insight_engine'] = intents_ops['id'].isin(['insight_request'])
intents_ops['hypothesis_mode'] = intents_ops['id'].isin(['hypothesis_request'])

intent_view = intents_ops[[
    'id', 'display_name', 'support_level',
    'required_params_text', 'optional_params_text',
    'future_function_name', 'requires_visualization',
    'direct_answer_possible', 'requires_clarification_by_default',
    'requires_insight_engine', 'hypothesis_mode',
]].rename(columns={'id': 'intent'})

display(intent_view)

,intent,display_name,support_level,required_params_text,optional_params_text,future_function_name,requires_visualization,direct_answer_possible,requires_clarification_by_default,requires_insight_engine,hypothesis_mode
0,query,Consulta directa,bien,"metric_id, zone_key",week_offset,"get_metric_value(metric_id, zone_key, week_off...",True,True,False,False,False
1,aggregate,Agregación,bien,"metric_id, group_by","week_offset, aggregation_func, filter_condition","aggregate_metric(metric_id, group_by, week_off...",True,False,False,False,False
2,rank,Ranking,bien,"metric_id, entity_level","week_offset, n, direction, filter_condition","rank_by_metric(metric_id, entity_level, week_o...",True,False,False,False,False
3,trend,Tendencia temporal,bien,"metric_id, zone_key",offsets,"get_trend(metric_id, zone_key, offsets)",True,False,False,False,False
4,compare,Comparación entre segmentos,bien,"metric_id, segment_a, segment_b",week_offset,"compare_segments(metric_id, segment_a, segment...",True,False,False,False,False
5,multivariable_filter,Filtro multivariable,parcial,conditions,"week_offset, entity_level","screen_by_conditions(conditions, week_offset, ...",True,False,True,False,False
6,insight_request,Request de insight automático,parcial,zone_key,"metrics, detectors, week_offset","run_insight_detectors(zone_key, metrics, detec...",True,False,False,True,False
7,hypothesis_request,Hipótesis explicativa,parcial,"outcome_metric, zone_key","candidate_metrics, week_offset","generate_hypothesis(outcome_metric, zone_key, ...",True,False,True,False,True


## 3) Diseño del planner estructurado

El planner transforma texto libre a un plan ejecutable y validable antes de llamar herramientas.

In [3]:
planner_schema = {
    'title': 'ChatbotExecutionPlan',
    'type': 'object',
    'required': ['intent', 'requested_output', 'requires_clarification', 'confidence_mode'],
    'properties': {
        'intent': {'type': 'string', 'description': 'Intent canonico de question_types'},
        'metric': {'type': ['string', 'null']},
        'entity_scope': {
            'type': 'object',
            'properties': {
                'country': {'type': ['string', 'null']},
                'city': {'type': ['string', 'null']},
                'zone': {'type': ['string', 'null']},
                'segment': {'type': ['string', 'null']},
            },
            'additionalProperties': False,
        },
        'filters': {'type': 'array', 'items': {'type': 'object'}},
        'comparison': {'type': ['object', 'null']},
        'time_window': {'type': ['string', 'null'], 'description': 'ej: L0W, L8W-L0W'},
        'requested_output': {'type': 'string', 'enum': ['kpi', 'table', 'chart', 'narrative', 'insights']},
        'chart_preference': {'type': ['string', 'null']},
        'confidence_mode': {'type': 'string', 'enum': ['strict', 'balanced', 'exploratory']},
        'requires_clarification': {'type': 'boolean'},
        'clarification_question': {'type': ['string', 'null']},
        'tool_calls': {'type': 'array', 'items': {'type': 'object'}},
    },
    'additionalProperties': False,
}

planner_examples = pd.DataFrame([
    {
        'user_question': 'Compara Perfect Orders entre Wealthy y Non Wealthy en Mexico',
        'intent': 'compare',
        'metric': 'perfect_orders',
        'entity_scope': {'country': 'MX', 'segment': 'ZONE_TYPE'},
        'time_window': 'L0W',
        'requested_output': 'table',
        'requires_clarification': False,
    },
    {
        'user_question': 'Que explica el crecimiento de orders en Chapinero?',
        'intent': 'hypothesis_request',
        'metric': 'orders_total',
        'entity_scope': {'country': 'CO', 'city': 'Bogota', 'zone': 'Chapinero'},
        'time_window': 'L8W-L0W',
        'requested_output': 'narrative',
        'requires_clarification': False,
    },
    {
        'user_question': 'Top 5 zonas con mayor lead penetration',
        'intent': 'rank',
        'metric': 'lead_penetration',
        'entity_scope': {},
        'time_window': 'L0W',
        'requested_output': 'table',
        'requires_clarification': True,
    },
])

display(pd.DataFrame([{'schema': json.dumps(planner_schema, ensure_ascii=False)[:500] + '...'}]))
display(planner_examples)

,schema
0,"{""title"": ""ChatbotExecutionPlan"", ""type"": ""obj..."


,user_question,intent,metric,entity_scope,time_window,requested_output,requires_clarification
0,Compara Perfect Orders entre Wealthy y Non Wea...,compare,perfect_orders,"{'country': 'MX', 'segment': 'ZONE_TYPE'}",L0W,table,False
1,Que explica el crecimiento de orders en Chapin...,hypothesis_request,orders_total,"{'country': 'CO', 'city': 'Bogota', 'zone': 'C...",L8W-L0W,narrative,False
2,Top 5 zonas con mayor lead penetration,rank,lead_penetration,{},L0W,table,True


In [ ]:
# ── validate_plan: semantic contract enforcement on the plan ──────────────
import yaml as _yaml

def validate_plan(plan: dict, metrics_cfg: list, biz_rules: dict, question_types_cfg: list) -> dict:
    """
    Apply semantic contract rules to a parsed plan before execution.
    Returns: {'valid': bool, 'action': 'execute'|'clarify'|'reject', 'reason': str, 'suggestion': str}
    """
    metric_id   = plan.get('metric')
    entity      = plan.get('entity_scope', {})
    intent      = plan.get('intent', '')
    time_window = plan.get('time_window', 'L0W')

    metrics_by_id = {m['id']: m for m in metrics_cfg}
    intents_by_id = {q['id']: q for q in question_types_cfg}
    not_comparable = biz_rules.get('peer_groups', {}).get('not_comparable', [])

    # 1. Unknown intent → reject
    if intent and intent not in intents_by_id and intent not in ('follow_up_scope_refine', 'follow_up_visualization'):
        return {'valid': False, 'action': 'reject',
                'reason': f'Intent "{intent}" no reconocido en question_types.yaml',
                'suggestion': f'Intents válidos: {list(intents_by_id.keys())}'}

    # 2. Suspended metric → reject + propose alternative
    if metric_id and metric_id in metrics_by_id:
        m = metrics_by_id[metric_id]
        if m.get('validation_status') == 'suspended_pending_definition':
            return {'valid': False, 'action': 'clarify',
                    'reason': f'Métrica "{metric_id}" suspendida: {str(m.get("validation_reason",""))[:100]}',
                    'suggestion': 'Sugiere métricas alternativas: perfect_orders, pro_adoption_last_week, restaurants_ss_atc_cvr'}

    # 3. lead_penetration in rank → reject
    if intent == 'rank' and metric_id == 'lead_penetration':
        return {'valid': False, 'action': 'reject',
                'reason': 'lead_penetration excluida de rankings por outlier extremo (máx=393.9)',
                'suggestion': 'Usa perfect_orders o pro_adoption_last_week para ranking'}

    # 4. ZONE ambiguity without COUNTRY or CITY → clarify
    zone = entity.get('zone')
    country = entity.get('country')
    city = entity.get('city')
    if zone and not country and not city:
        return {'valid': False, 'action': 'clarify',
                'reason': f'ZONE="{zone}" no es único — el mismo nombre existe en múltiples ciudades y países.',
                'suggestion': 'Pregunta: ¿En qué país y ciudad? Ej: "¿Te refieres a Bogotá, Colombia?"'}

    # 5. Hypothesis without framing → enforce caveat
    if intent == 'hypothesis_request' and not plan.get('requires_clarification', False):
        return {'valid': True, 'action': 'execute',
                'reason': '',
                'suggestion': 'MANDATORY: incluir caveat de no-causalidad. Usar lenguaje: "posible driver", "asociación observada"'}

    # 6. Metric provisional → add caveat but allow
    if metric_id and metric_id in metrics_by_id:
        m = metrics_by_id[metric_id]
        if m.get('validation_status') in ('provisional', 'pending_business_validation'):
            return {'valid': True, 'action': 'execute',
                    'reason': '',
                    'suggestion': f'Incluir caveat: dirección de {m["display_name"]} es provisional — no validada con negocio.'}

    return {'valid': True, 'action': 'execute', 'reason': '', 'suggestion': ''}


# ── demonstrate validation on example plans ────────────────────────────────
test_plans = [
    {'label': 'Lead Penetration ranking (suspended)',
     'plan': {'intent': 'rank', 'metric': 'lead_penetration', 'entity_scope': {}, 'time_window': 'L0W', 'requested_output': 'table', 'requires_clarification': False}},
    {'label': 'Perfect Orders trend — zone ambiguous',
     'plan': {'intent': 'trend', 'metric': 'perfect_orders', 'entity_scope': {'zone': 'Chapinero'}, 'time_window': 'L8W-L0W', 'requested_output': 'chart', 'requires_clarification': False}},
    {'label': 'Hypothesis request (valid)',
     'plan': {'intent': 'hypothesis_request', 'metric': 'orders_total', 'entity_scope': {'country': 'CO', 'city': 'Bogota', 'zone': 'Chapinero'}, 'time_window': 'L8W-L0W', 'requested_output': 'narrative', 'requires_clarification': False}},
    {'label': 'Compare segments — valid',
     'plan': {'intent': 'compare', 'metric': 'perfect_orders', 'entity_scope': {'country': 'MX', 'segment': 'ZONE_TYPE'}, 'time_window': 'L0W', 'requested_output': 'table', 'requires_clarification': False}},
]

results = []
for t in test_plans:
    v = validate_plan(
        t['plan'],
        metrics_cfg.get('metrics', []),
        business_rules,
        question_types_cfg.get('question_types', []),
    )
    results.append({'label': t['label'], **v})

validation_demo = pd.DataFrame(results)
print('=== validate_plan() demo ===')
for _, row in validation_demo.iterrows():
    icon = '✓' if row['valid'] else '✗'
    print(f'  [{icon}] [{row["action"]:8}] {row["label"]}')
    if row['reason']:
        print(f'           Reason: {row["reason"][:100]}')
    if row['suggestion']:
        print(f'           Suggestion: {row["suggestion"][:100]}')
print()

## 4) Reglas de clarificacion

El chatbot no debe asumir parametros ambiguos ni ejecutar comparaciones invalidas segun NB20.

In [4]:
clarification_rules = pd.DataFrame([
    {'trigger': 'metrica_ambigua', 'why': 'No se puede ejecutar funcion sin metrica canonica', 'good_clarification': 'Que metrica quieres analizar? Ej: Perfect Orders o Gross Profit UE.'},
    {'trigger': 'entidad_ambigua', 'why': 'ZONE no es unica sin COUNTRY y CITY', 'good_clarification': 'Te refieres a que pais y ciudad para esa zona?'},
    {'trigger': 'comparacion_invalida', 'why': 'NB20 define pares no comparables', 'good_clarification': 'Esa comparacion no es valida. Quieres comparar dentro del mismo pais y tipo de zona?'},
    {'trigger': 'peer_group_debil', 'why': 'Benchmark con n pequeno es fragil', 'good_clarification': 'El peer group es pequeno. Continuo con caveat o prefieres ampliar el alcance?'},
    {'trigger': 'causalidad_fuerte', 'why': 'El sistema solo soporta asociaciones no causales', 'good_clarification': 'Puedo darte posibles drivers asociados, no causalidad confirmada. Continuo?'},
    {'trigger': 'fuera_de_alcance', 'why': 'Intent no cubierto por funciones deterministicas', 'good_clarification': 'No puedo responder eso con el contrato actual. Te propongo reformular en formato ranking, tendencia, comparacion o insight.'},
    {'trigger': 'metrica_suspendida', 'why': 'validation_status suspendida impide uso analitico confiable', 'good_clarification': 'Esa metrica esta suspendida temporalmente. Quieres usar una metrica alternativa?'},
])

display(clarification_rules)

,trigger,why,good_clarification
0,metrica_ambigua,No se puede ejecutar funcion sin metrica canonica,Que metrica quieres analizar? Ej: Perfect Orde...
1,entidad_ambigua,ZONE no es unica sin COUNTRY y CITY,Te refieres a que pais y ciudad para esa zona?
2,comparacion_invalida,NB20 define pares no comparables,Esa comparacion no es valida. Quieres comparar...
3,peer_group_debil,Benchmark con n pequeno es fragil,El peer group es pequeno. Continuo con caveat ...
4,causalidad_fuerte,El sistema solo soporta asociaciones no causales,"Puedo darte posibles drivers asociados, no cau..."
5,fuera_de_alcance,Intent no cubierto por funciones deterministicas,No puedo responder eso con el contrato actual....
6,metrica_suspendida,validation_status suspendida impide uso analit...,Esa metrica esta suspendida temporalmente. Qui...


## 5) Gestion de contexto conversacional

Se define estado minimo, seguro y util para follow-ups sin memoria libre peligrosa.

In [5]:
chat_state_schema = {
    'title': 'ChatSessionState',
    'type': 'object',
    'properties': {
        'last_metric_id': {'type': ['string', 'null']},
        'last_entity': {'type': ['object', 'null']},
        'last_filters': {'type': 'array', 'items': {'type': 'object'}},
        'last_comparison': {'type': ['object', 'null']},
        'last_time_window': {'type': ['string', 'null']},
        'mode': {'type': 'string', 'enum': ['direct_answer', 'hypothesis_mode']},
        'last_visualization': {'type': ['string', 'null']},
    },
    'required': ['mode', 'last_filters'],
    'additionalProperties': False,
}

context_policy = pd.DataFrame([
    {'keep': 'last_metric_id', 'reason': 'permite resolver follow-up tipo "y en Colombia?"'},
    {'keep': 'last_entity', 'reason': 'evita pedir entidad completa en cada turno'},
    {'keep': 'last_filters', 'reason': 'mantiene alcance analitico trazable'},
    {'keep': 'last_comparison', 'reason': 'preserva baseline de comparacion'},
    {'keep': 'last_time_window', 'reason': 'evita cambios silenciosos de periodo'},
    {'keep': 'mode', 'reason': 'separa respuesta directa de hypothesis mode'},
    {'do_not_keep': 'supuestos_no_confirmados', 'reason': 'evita memoria inventada'},
    {'do_not_keep': 'narrativas_largas', 'reason': 'reduce arrastre de contexto irrelevante'},
])

display(context_policy.fillna(''))

,keep,reason,do_not_keep
0,last_metric_id,"permite resolver follow-up tipo ""y en Colombia?""",
1,last_entity,evita pedir entidad completa en cada turno,
2,last_filters,mantiene alcance analitico trazable,
3,last_comparison,preserva baseline de comparacion,
4,last_time_window,evita cambios silenciosos de periodo,
5,mode,separa respuesta directa de hypothesis mode,
6,,evita memoria inventada,supuestos_no_confirmados
7,,reduce arrastre de contexto irrelevante,narrativas_largas


## 6) Contrato de herramientas / funciones

Las funciones son deterministicas y ejecutan el calculo. El chatbot solo planifica, valida y presenta resultados.

In [6]:
tool_contract = pd.DataFrame([
    {'tool': 'get_metric_value', 'input_schema': 'metric_id, entity_scope, week_offset', 'output_schema': 'value, metadata, caveats', 'errors': 'metric_not_found|entity_not_found|insufficient_data', 'chatbot_caveat': 'direction/validation caveats'},
    {'tool': 'aggregate_metric', 'input_schema': 'metric_id, group_by, filters, week_offset, agg_func', 'output_schema': 'rows[group, value, n, coverage]', 'errors': 'invalid_group_by|unsupported_metric', 'chatbot_caveat': 'coverage and excluded zones'},
    {'tool': 'rank_by_metric', 'input_schema': 'metric_id, entity_level, n, direction, filters', 'output_schema': 'ranked_rows', 'errors': 'metric_not_rankable|direction_mismatch', 'chatbot_caveat': 'invert when lower_is_better'},
    {'tool': 'get_trend', 'input_schema': 'metric_id, entity_scope, window', 'output_schema': 'series[value,wow]', 'errors': 'insufficient_history', 'chatbot_caveat': 'no complex forecasting with 9 weeks'},
    {'tool': 'compare_segments', 'input_schema': 'metric_id, segment_a, segment_b, week_offset', 'output_schema': 'comparison_table + delta', 'errors': 'invalid_comparison', 'chatbot_caveat': 'respect not_comparable rules'},
    {'tool': 'screen_by_conditions', 'input_schema': 'conditions[], entity_level, week_offset', 'output_schema': 'matching_entities', 'errors': 'missing_threshold|unsupported_condition', 'chatbot_caveat': 'thresholds must be explicit'},
    {'tool': 'run_insight_detectors', 'input_schema': 'entity_scope, metrics, detectors, week_offset', 'output_schema': 'insight_candidates[]', 'errors': 'detector_not_available', 'chatbot_caveat': 'provisional thresholds caveat'},
    {'tool': 'get_insight_candidates', 'input_schema': 'filters, sort, limit', 'output_schema': 'insight_table', 'errors': 'empty_result', 'chatbot_caveat': 'report confidence and caveats'},
    {'tool': 'generate_hypothesis_candidates', 'input_schema': 'target_metric, entity_scope, candidate_metrics', 'output_schema': 'association_table', 'errors': 'insufficient_points|target_not_supported', 'chatbot_caveat': 'association not causation'},
    {'tool': 'render_chart_data', 'input_schema': 'intent, data_payload, chart_pref', 'output_schema': 'chart_spec', 'errors': 'chart_not_supported', 'chatbot_caveat': 'chart is view, not evidence source'},
])

display(tool_contract)

,tool,input_schema,output_schema,errors,chatbot_caveat
0,get_metric_value,"metric_id, entity_scope, week_offset","value, metadata, caveats",metric_not_found|entity_not_found|insufficient...,direction/validation caveats
1,aggregate_metric,"metric_id, group_by, filters, week_offset, agg...","rows[group, value, n, coverage]",invalid_group_by|unsupported_metric,coverage and excluded zones
2,rank_by_metric,"metric_id, entity_level, n, direction, filters",ranked_rows,metric_not_rankable|direction_mismatch,invert when lower_is_better
3,get_trend,"metric_id, entity_scope, window","series[value,wow]",insufficient_history,no complex forecasting with 9 weeks
4,compare_segments,"metric_id, segment_a, segment_b, week_offset",comparison_table + delta,invalid_comparison,respect not_comparable rules
5,screen_by_conditions,"conditions[], entity_level, week_offset",matching_entities,missing_threshold|unsupported_condition,thresholds must be explicit
6,run_insight_detectors,"entity_scope, metrics, detectors, week_offset",insight_candidates[],detector_not_available,provisional thresholds caveat
7,get_insight_candidates,"filters, sort, limit",insight_table,empty_result,report confidence and caveats
8,generate_hypothesis_candidates,"target_metric, entity_scope, candidate_metrics",association_table,insufficient_points|target_not_supported,association not causation
9,render_chart_data,"intent, data_payload, chart_pref",chart_spec,chart_not_supported,"chart is view, not evidence source"


In [ ]:
# ── How insight_request routes to NB30 outputs ───────────────────────────
# NB30 generates insight_candidates.parquet — this is the authoritative source.
# The chatbot does NOT re-run detectors; it queries the pre-computed candidates.

if not insight_candidates.empty:
    ic = insight_candidates.copy()

    # ── show schema ────────────────────────────────────────────────────────
    print('insight_candidates schema:')
    print(ic.dtypes.to_string())
    print(f'Shape: {ic.shape}')
    print()

    # ── routing function: insight_request intent → NB30 lookup ───────────
    def route_insight_request(
        entity_scope: dict,
        week_offset: str = 'L0W',
        top_n: int = 5,
        min_confidence: str = 'low_confidence',
        candidates: 'pd.DataFrame' = ic,
    ) -> 'pd.DataFrame':
        """
        Given entity_scope from planner, return top-N ranked insights.
        This is what the chatbot calls for insight_request intent.
        """
        df = candidates.copy()

        # filter by entity scope
        if entity_scope.get('country'):
            df = df[df['COUNTRY'].str.upper() == entity_scope['country'].upper()]
        if entity_scope.get('city'):
            df = df[df['CITY'].str.lower() == entity_scope['city'].lower()]
        if entity_scope.get('zone'):
            df = df[df['ZONE'].str.lower() == entity_scope['zone'].lower()]

        # exclude suspended metrics
        suspended_metrics = [
            m['display_name']
            for m in metrics_cfg.get('metrics', [])
            if m.get('validation_status') == 'suspended_pending_definition'
        ]
        if 'metric_display_name' in df.columns and suspended_metrics:
            df = df[~df['metric_display_name'].isin(suspended_metrics)]
        elif 'id' in df.columns and suspended_metrics:
            df = df[~df['id'].isin(suspended_metrics)]

        # sort by score
        score_col = 'final_rank_score' if 'final_rank_score' in df.columns else (
            'severity_score' if 'severity_score' in df.columns else df.columns[-1])
        df = df.sort_values(score_col, ascending=False)

        return df.head(top_n)

    # ── example: top insights for MX ──────────────────────────────────────
    top_mx = route_insight_request({'country': 'MX'}, top_n=5)
    print('Top 5 insights for MX (example routing):')
    disp_cols = [c for c in ['COUNTRY','CITY','ZONE','insight_category','detector_name','narrative_short',
                              'final_rank_score','severity_score'] if c in top_mx.columns]
    print(top_mx[disp_cols].to_string(index=False) if not top_mx.empty else '(no data for MX in candidates)')
    print()

    # ── show chatbot response format for insight_request ─────────────────
    print('=== Expected chatbot response format for insight_request ===')
    print('short_answer : "Encontré X alertas activas para [entity] en L0W."')
    print('evidence     : tabla insight_candidates filtrada, ordenada por final_rank_score')
    print('scope        : entity_scope + week_offset + n_candidates')
    print('viz          : alert_cards (una card por insight, ordenadas por severidad)')
    print('caveat       : "Thresholds provisionales — tasa de falsos positivos sin calibrar."')
    print('next_q       : "¿Quieres ver la tendencia de alguna de estas métricas? | ¿Filtrar por severidad?"')
else:
    print('insight_candidates.parquet no encontrado — ejecuta NB30 primero.')
    print('El chatbot cargará este artefacto en runtime.')

## 7) Contrato de respuesta del chatbot

Formato fijo por respuesta para evitar texto libre desordenado:
1. respuesta corta
2. evidencia principal
3. filtros/alcance usados
4. visualizacion sugerida o incluida
5. caveat/limite
6. siguientes preguntas sugeridas

In [7]:
response_contract = pd.DataFrame([
    {'intent': 'query', 'short_answer': 'valor puntual', 'evidence': 'valor + semana + fuente', 'scope': 'entity + metric + offset', 'viz': 'kpi_card', 'caveat': 'validation/direction', 'next_questions': 'trend|compare'},
    {'intent': 'aggregate', 'short_answer': 'estadistico principal', 'evidence': 'tabla agregada + n', 'scope': 'group_by + filtros', 'viz': 'bar_chart', 'caveat': 'coverage/missing', 'next_questions': 'rank|segment drill-down'},
    {'intent': 'rank', 'short_answer': 'top/bottom n', 'evidence': 'tabla rankeada', 'scope': 'entity_level + n + filtros', 'viz': 'ranked_table', 'caveat': 'lower_is_better + elegibilidad', 'next_questions': 'why this zone?|trend'},
    {'intent': 'trend', 'short_answer': 'direccion reciente', 'evidence': 'serie + wow', 'scope': 'entity + ventana', 'viz': 'line_chart', 'caveat': '9 weeks no forecasting', 'next_questions': 'insight_request'},
    {'intent': 'compare', 'short_answer': 'delta entre segmentos', 'evidence': 'tabla comparativa + n', 'scope': 'segment_a vs segment_b', 'viz': 'side_by_side_bar', 'caveat': 'comparabilidad', 'next_questions': 'rank within segment'},
    {'intent': 'insight_request', 'short_answer': 'top hallazgos', 'evidence': 'insight_candidates ordenados', 'scope': 'entity/filtro + detectores', 'viz': 'alert_cards', 'caveat': 'thresholds provisionales', 'next_questions': 'show evidence|actions'},
    {'intent': 'hypothesis_request', 'short_answer': 'posibles drivers', 'evidence': 'asociaciones y caveats', 'scope': 'target_metric + entidad', 'viz': 'correlation_table', 'caveat': 'no causalidad', 'next_questions': 'validate with ops context'},
])

display(response_contract)

,intent,short_answer,evidence,scope,viz,caveat,next_questions
0,query,valor puntual,valor + semana + fuente,entity + metric + offset,kpi_card,validation/direction,trend|compare
1,aggregate,estadistico principal,tabla agregada + n,group_by + filtros,bar_chart,coverage/missing,rank|segment drill-down
2,rank,top/bottom n,tabla rankeada,entity_level + n + filtros,ranked_table,lower_is_better + elegibilidad,why this zone?|trend
3,trend,direccion reciente,serie + wow,entity + ventana,line_chart,9 weeks no forecasting,insight_request
4,compare,delta entre segmentos,tabla comparativa + n,segment_a vs segment_b,side_by_side_bar,comparabilidad,rank within segment
5,insight_request,top hallazgos,insight_candidates ordenados,entity/filtro + detectores,alert_cards,thresholds provisionales,show evidence|actions
6,hypothesis_request,posibles drivers,asociaciones y caveats,target_metric + entidad,correlation_table,no causalidad,validate with ops context


## 8) Reglas de lenguaje y seguridad analitica

Se operacionalizan reglas de NB20 para respuestas conversacionales consistentes y seguras.

In [8]:
language_ops = pd.DataFrame([
    {'rule': 'mejoro/empeoro', 'when_allowed': 'solo con desired_direction definida', 'required_caveat': 'direction provisional si aplica'},
    {'rule': 'possible_driver', 'when_allowed': 'solo en intent hypothesis_request', 'required_caveat': 'association_not_causation'},
    {'rule': 'causality_guard', 'when_allowed': 'si usuario pide causalidad', 'required_caveat': 'rechazar causalidad fuerte y ofrecer hipotesis'},
    {'rule': 'peer_weak', 'when_allowed': 'peer_confidence low_confidence', 'required_caveat': 'benchmark fragil por n pequeno'},
    {'rule': 'metric_pending_validation', 'when_allowed': 'validation_status pending', 'required_caveat': 'resultado sujeto a validacion negocio'},
    {'rule': 'metric_suspended', 'when_allowed': 'validation_status suspended', 'required_caveat': 'no usar; proponer alternativa'},
])

display(language_ops)

,rule,when_allowed,required_caveat
0,mejoro/empeoro,solo con desired_direction definida,direction provisional si aplica
1,possible_driver,solo en intent hypothesis_request,association_not_causation
2,causality_guard,si usuario pide causalidad,rechazar causalidad fuerte y ofrecer hipotesis
3,peer_weak,peer_confidence low_confidence,benchmark fragil por n pequeno
4,metric_pending_validation,validation_status pending,resultado sujeto a validacion negocio
5,metric_suspended,validation_status suspended,no usar; proponer alternativa


In [ ]:
# ── Operational language guard functions ─────────────────────────────────
# These are the rules from NB20.language_rules made executable.
# NB40 (chatbot) applies these before rendering any narrative.

def apply_direction_guard(metric_id: str, value_delta: float, metrics_cfg: list) -> str:
    """
    Returns phrasing: 'aumentó/disminuyó' (neutral) or 'mejoró/empeoró' (directional).
    Never uses directional language for provisional/pending metrics.
    """
    m_map = {m['id']: m for m in metrics_cfg}
    m = m_map.get(metric_id, {})
    direction = m.get('desired_direction', 'unknown_pending_validation')
    confidence = m.get('direction_confidence', 'provisional')
    status = m.get('validation_status', 'pending_business_validation')

    if status == 'suspended_pending_definition':
        return f'cambió en {value_delta:+.3f} (métrica suspendida — no usar dirección)'

    direction_known = direction in ('higher_is_better', 'lower_is_better')
    if not direction_known or confidence == 'unknown_pending_validation':
        verb = 'aumentó' if value_delta > 0 else 'disminuyó'
        return f'{verb} en {abs(value_delta):.3f}'

    # direction known (provisional)
    is_improvement = (direction == 'higher_is_better' and value_delta > 0) or                      (direction == 'lower_is_better' and value_delta < 0)
    verb = 'mejoró' if is_improvement else 'empeoró'
    caveat = ' *(dirección provisional — no validada con negocio)*' if confidence == 'provisional' else ''
    return f'{verb} en {abs(value_delta):.3f}{caveat}'


def apply_hypothesis_guard(text: str) -> str:
    """Reject causal framing, enforce associative framing."""
    causal_terms = business_rules.get('language_rules', {}).get('causal_language', {}).get('forbidden_terms', [])
    found = [t for t in causal_terms if t in text.lower()]
    if found:
        return (
            f'[RECHAZO_CAUSALIDAD] El texto contiene términos causales no permitidos: {found}. '
            'Reformular usando: "se asocia con", "covaría con", "es consistente con".'
        )
    return text


def build_uncertainty_caveat(metric_id: str, peer_n: int | None, metrics_cfg: list, biz_rules: dict) -> str:
    """Build an uncertainty caveat string for a given metric and peer context."""
    m_map = {m['id']: m for m in metrics_cfg}
    m = m_map.get(metric_id, {})
    min_size = biz_rules.get('peer_groups', {}).get('size_rules', {}).get('min_size_for_benchmark', 10)
    warn_size = biz_rules.get('peer_groups', {}).get('size_rules', {}).get('min_size_warning_threshold', 5)

    caveats = []
    if m.get('direction_confidence') == 'provisional':
        caveats.append(f'Dirección de {m.get("display_name", metric_id)} provisional — no validada con negocio.')
    if m.get('validation_status') == 'suspended_pending_definition':
        caveats.append(f'Métrica suspendida — no usar en análisis comparativos.')
    if m.get('outlier_risk') == 'high':
        caveats.append(f'Alta variabilidad ({m.get("display_name","")}) — usar MAD/IQR, no STD.')
    if peer_n is not None:
        if peer_n < warn_size:
            caveats.append(f'Peer group muy pequeño (n={peer_n}) — benchmark no disponible.')
        elif peer_n < min_size:
            caveats.append(f'Peer group pequeño (n={peer_n}) — benchmark low_confidence.')
    return ' | '.join(caveats) if caveats else 'Sin caveats adicionales para esta respuesta.'


# ── demonstrate guards ─────────────────────────────────────────────────────
print('=== apply_direction_guard() demos ===')
test_cases = [
    ('gross_profit_ue',        +0.8),
    ('restaurants_markdowns_gmv', +0.02),
    ('lead_penetration',       +5.0),
    ('perfect_orders',         -0.03),
]
for mid, delta in test_cases:
    result = apply_direction_guard(mid, delta, metrics_cfg.get('metrics', []))
    print(f'  {mid:45} Δ={delta:+.3f} → {result}')

print()
print('=== apply_hypothesis_guard() demos ===')
test_texts = [
    'El CVR causó el crecimiento de órdenes.',
    'El CVR se asocia con el crecimiento de órdenes (ρ=0.23, n=9 semanas).',
]
for t in test_texts:
    print(f'  IN:  {t}')
    print(f'  OUT: {apply_hypothesis_guard(t)}')
    print()

print('=== build_uncertainty_caveat() demos ===')
for mid, n in [('perfect_orders', 25), ('gross_profit_ue', 7), ('lead_penetration', 15)]:
    c = build_uncertainty_caveat(mid, n, metrics_cfg.get('metrics', []), business_rules)
    print(f'  {mid:45} n={n}: {c}')

## 9) UX conversacional propuesta

Wireframe logico de respuesta por tipo de pregunta.

- Header: pregunta normalizada + alcance
- Body: respuesta corta + evidencia + tabla/grafico
- Footer: caveats + prompts sugeridos

In [9]:
ux_components = pd.DataFrame([
    {'intent': 'query', 'primary_component': 'kpi_card', 'secondary_component': 'peer_context_chip', 'suggested_prompts': 'Ver tendencia | Comparar vs peers'},
    {'intent': 'compare', 'primary_component': 'comparison_table', 'secondary_component': 'delta_bar', 'suggested_prompts': 'Solo en Colombia | Mostrar top gaps'},
    {'intent': 'trend', 'primary_component': 'line_chart', 'secondary_component': 'wow_table', 'suggested_prompts': 'Mostrar solo ultimas 4 semanas | Activar alertas'},
    {'intent': 'insight_request', 'primary_component': 'alert_cards', 'secondary_component': 'evidence_panel', 'suggested_prompts': 'Ver evidencia | Priorizar por severidad'},
    {'intent': 'hypothesis_request', 'primary_component': 'association_table', 'secondary_component': 'caveat_panel', 'suggested_prompts': 'Ver solo asociaciones fuertes | Mostrar no causalidad'},
])

wireframe_text = textwrap.dedent('''
[Conversation Header]
  - User query normalized
  - Active scope chips (metric, entity, period)

[Primary Insight Block]
  - Short answer
  - Main evidence table/chart

[Governance Block]
  - Caveats (validation, peer confidence, causality guard)
  - Tool trace (optional for audit mode)

[Follow-up Block]
  - Suggested prompts
''')

display(ux_components)
print(wireframe_text)

,intent,primary_component,secondary_component,suggested_prompts
0,query,kpi_card,peer_context_chip,Ver tendencia | Comparar vs peers
1,compare,comparison_table,delta_bar,Solo en Colombia | Mostrar top gaps
2,trend,line_chart,wow_table,Mostrar solo ultimas 4 semanas | Activar alertas
3,insight_request,alert_cards,evidence_panel,Ver evidencia | Priorizar por severidad
4,hypothesis_request,association_table,caveat_panel,Ver solo asociaciones fuertes | Mostrar no cau...



[Conversation Header]
  - User query normalized
  - Active scope chips (metric, entity, period)

[Primary Insight Block]
  - Short answer
  - Main evidence table/chart

[Governance Block]
  - Caveats (validation, peer confidence, causality guard)
  - Tool trace (optional for audit mode)

[Follow-up Block]
  - Suggested prompts



## 10) Golden conversation flows

Se definen flujos esperados (intent -> herramientas -> respuesta -> caveat) para pruebas end-to-end del orquestador.

In [10]:
golden_flows = pd.DataFrame([
    {'flow_id': 'GF01', 'user_query': 'Top 5 zonas con mayor Lead Penetration', 'intent': 'rank', 'tools': 'rank_by_metric', 'requires_clarification': True, 'ideal_response': 'rechazar metrica suspendida y proponer alternativa', 'expected_caveat': 'metric_suspended'},
    {'flow_id': 'GF02', 'user_query': 'Compara Perfect Orders entre Wealthy y Non Wealthy en Mexico', 'intent': 'compare', 'tools': 'compare_segments', 'requires_clarification': False, 'ideal_response': 'tabla comparativa con delta', 'expected_caveat': 'direction provisional'},
    {'flow_id': 'GF03', 'user_query': 'Evolucion de Gross Profit UE en Chapinero', 'intent': 'trend', 'tools': 'get_trend, render_chart_data', 'requires_clarification': True, 'ideal_response': 'pedir ciudad/pais si falta', 'expected_caveat': 'no forecasting con 9 semanas'},
    {'flow_id': 'GF04', 'user_query': 'Promedio de Lead Penetration por pais', 'intent': 'aggregate', 'tools': 'aggregate_metric', 'requires_clarification': True, 'ideal_response': 'rechazar uso por suspension', 'expected_caveat': 'metric_suspended'},
    {'flow_id': 'GF05', 'user_query': 'Zonas con alto Lead Penetration y bajo Perfect Orders', 'intent': 'multivariable_filter', 'tools': 'screen_by_conditions', 'requires_clarification': True, 'ideal_response': 'pedir reemplazo de metrica suspendida y thresholds', 'expected_caveat': 'unsupported_condition'},
    {'flow_id': 'GF06', 'user_query': 'Zonas que mas crecen en orders y que podria explicarlo', 'intent': 'hypothesis_request', 'tools': 'get_insight_candidates, generate_hypothesis_candidates', 'requires_clarification': False, 'ideal_response': 'drivers asociados no causales', 'expected_caveat': 'association_not_causation'},
    {'flow_id': 'GF07', 'user_query': 'Y solo en Colombia?', 'intent': 'follow_up_scope_refine', 'tools': 'reuse_last_tool_with_filter', 'requires_clarification': False, 'ideal_response': 'mantener metrica e intent previos filtrando country=CO', 'expected_caveat': 'preserve prior context'},
    {'flow_id': 'GF08', 'user_query': 'Muestralo en grafico', 'intent': 'follow_up_visualization', 'tools': 'render_chart_data', 'requires_clarification': False, 'ideal_response': 'transformar ultima respuesta a chart', 'expected_caveat': 'chart reflects deterministic output'},
    {'flow_id': 'GF09', 'user_query': 'Que problemas tiene Bogota esta semana?', 'intent': 'insight_request', 'tools': 'run_insight_detectors', 'requires_clarification': False, 'ideal_response': 'top insights por final_rank_score', 'expected_caveat': 'thresholds provisionales'},
    {'flow_id': 'GF10', 'user_query': 'Demuestrame que Restaurants SS > ATC CVR causa mas orders', 'intent': 'hypothesis_request', 'tools': 'generate_hypothesis_candidates', 'requires_clarification': True, 'ideal_response': 'rechazar causalidad fuerte, ofrecer evidencia asociativa', 'expected_caveat': 'no causal claims'},
])

display(golden_flows)

,flow_id,user_query,intent,tools,requires_clarification,ideal_response,expected_caveat
0,GF01,Top 5 zonas con mayor Lead Penetration,rank,rank_by_metric,True,rechazar metrica suspendida y proponer alterna...,metric_suspended
1,GF02,Compara Perfect Orders entre Wealthy y Non Wea...,compare,compare_segments,False,tabla comparativa con delta,direction provisional
2,GF03,Evolucion de Gross Profit UE en Chapinero,trend,"get_trend, render_chart_data",True,pedir ciudad/pais si falta,no forecasting con 9 semanas
3,GF04,Promedio de Lead Penetration por pais,aggregate,aggregate_metric,True,rechazar uso por suspension,metric_suspended
4,GF05,Zonas con alto Lead Penetration y bajo Perfect...,multivariable_filter,screen_by_conditions,True,pedir reemplazo de metrica suspendida y thresh...,unsupported_condition
5,GF06,Zonas que mas crecen en orders y que podria ex...,hypothesis_request,"get_insight_candidates, generate_hypothesis_ca...",False,drivers asociados no causales,association_not_causation
6,GF07,Y solo en Colombia?,follow_up_scope_refine,reuse_last_tool_with_filter,False,mantener metrica e intent previos filtrando co...,preserve prior context
7,GF08,Muestralo en grafico,follow_up_visualization,render_chart_data,False,transformar ultima respuesta a chart,chart reflects deterministic output
8,GF09,Que problemas tiene Bogota esta semana?,insight_request,run_insight_detectors,False,top insights por final_rank_score,thresholds provisionales
9,GF10,Demuestrame que Restaurants SS > ATC CVR causa...,hypothesis_request,generate_hypothesis_candidates,True,"rechazar causalidad fuerte, ofrecer evidencia ...",no causal claims


### Golden flows — diálogos detallados

Los flujos a continuación muestran el turno completo: pregunta → plan → llamada a tool → respuesta + caveat.

---

#### GF01 — Métrica suspendida en ranking

```
USER:  Top 5 zonas con mayor Lead Penetration

PLAN:
  intent: rank
  metric: lead_penetration
  validation → REJECTED (suspended_pending_definition)
  action: clarify

BOT:   "Lead Penetration está temporalmente suspendida del sistema analítico.
        El denominador no está clarificado y el valor máximo observado es 393.9
        en una escala que debería ser ratio. No se puede generar un ranking confiable.
        ¿Quieres ver el ranking por Perfect Orders o Pro Adoption en su lugar?"

CAVEAT: metric_suspended
NEXT_Q: "Top 5 por Perfect Orders | Top 5 por Pro Adoption (Last Week Status)"
```

---

#### GF02 — Comparación de segmentos (válida)

```
USER:  Compara Perfect Orders entre Wealthy y Non Wealthy en México

PLAN:
  intent: compare
  metric: perfect_orders
  entity_scope: {country: MX, segment: ZONE_TYPE}
  time_window: L0W
  tool: compare_segments(metric_id='perfect_orders', segment_a='Wealthy', segment_b='Non Wealthy',
                         filters={'country':'MX'}, week_offset='L0W')

TOOL OUTPUT:
  segment=Wealthy    | value=0.834 | n_zones=34 | confidence=reliable
  segment=Non Wealthy| value=0.791 | n_zones=45 | confidence=reliable
  delta: Wealthy − Non Wealthy = +0.043

BOT:   "En México (L0W), las zonas Wealthy tienen una mediana de Perfect Orders
        de 0.834 frente a 0.791 en zonas Non Wealthy — diferencia de +0.043.
        Tanto Wealthy (n=34) como Non Wealthy (n=45) cuentan con grupos confiables."

CAVEAT: "Dirección de Perfect Orders provisional — no validada con negocio.
         La definición de 'perfecto' puede variar si los SLAs difieren entre mercados."
VIZ:   side_by_side_bar
NEXT_Q: "¿Ver tendencia de cada segmento? | ¿Filtrar solo zonas High Priority?"
```

---

#### GF03 — Trend con zona ambigua

```
USER:  Evolución de Gross Profit UE en Chapinero

PLAN:
  intent: trend
  metric: gross_profit_ue
  entity_scope: {zone: Chapinero}
  → ZONE sin COUNTRY y CITY → clarify

BOT:   "El nombre 'Chapinero' aparece en múltiples ciudades del dataset.
        ¿Te refieres a Chapinero en Bogotá, Colombia?"

USER:  Sí, Bogotá Colombia

PLAN (2º turno):
  intent: trend
  metric: gross_profit_ue
  entity_scope: {country: CO, city: Bogota, zone: Chapinero}
  tool: get_trend(metric_id='gross_profit_ue', zone_key='CO|Bogota|Chapinero', offsets=['L8W','L7W',...,'L0W'])

TOOL OUTPUT:
  L8W: -1.2 | L7W: -0.8 | L6W: -1.0 | L5W: -0.3 | L4W: 0.1 | L3W: 0.4 | L2W: 0.2 | L1W: 0.5 | L0W: 0.3
  WoW(L0W vs L1W): -0.2

BOT:   "Gross Profit UE en Bogotá-Chapinero mostró una recuperación entre L5W y L1W
        (pasó de -0.3 a +0.5). En L0W bajó levemente a +0.3 respecto a L1W (WoW = -0.2)."

CAVEAT: "Alta variabilidad (outlier_risk=high). Usar MAD para benchmarks — no STD.
         Con 9 semanas no se puede confirmar una tendencia estadísticamente robusta.
         Dirección provisional."
VIZ:   line_chart (L8W→L0W) con anotación WoW
NEXT_Q: "¿Comparar contra peer group? | ¿Ver Perfect Orders en la misma zona?"
```

---

#### GF06 — Hypothesis request (no-causal)

```
USER:  ¿Qué podría explicar el crecimiento de órdenes en zonas con alto CVR de restaurantes?

PLAN:
  intent: hypothesis_request
  metric: orders_total (outcome)
  candidate_metrics: all (correlaciones disponibles)
  entity_scope: {} (todas las zonas con alto restaurants_ss_atc_cvr)
  tool: generate_hypothesis_candidates(outcome_metric='orders_total',
                                       zone_filter={'high_restaurants_ss_atc_cvr': True})

TOOL OUTPUT (correlaciones Spearman WoW, n=9 semanas):
  restaurants_ss_atc_cvr: ρ=+0.229 (más alta del dataset)
  pro_adoption_last_week:  ρ=+0.10
  gross_profit_ue:         ρ=-0.042

BOT:   "Posibles drivers asociados con WoW de órdenes (hipótesis de trabajo,
        correlaciones Spearman n=9 semanas):

        1. Restaurants SS > ATC CVR: ρ=+0.229 — la asociación más alta del dataset.
           Las semanas con mayor CVR covaríen con más órdenes.
        2. Pro Adoption: ρ=+0.10 — asociación leve, positiva.

        ⚠️ Esto NO es evidencia causal. Son co-movimientos observados en 9 semanas.
        Útil para priorizar qué explorar con el equipo, no para tomar decisiones."

CAVEAT: "Correlaciones Spearman WoW con n=9 semanas — evidencia débil.
         No usar como base para cambios operacionales sin validación adicional."
VIZ:   correlation_table con ρ, n, framing_text
NEXT_Q: "¿Ver tendencia de CVR en zonas específicas? | ¿Comparar zonas con ρ alto vs bajo?"
```

---

#### GF09 — Insight request (NB30 routing)

```
USER:  ¿Qué problemas tiene Bogotá esta semana?

PLAN:
  intent: insight_request
  entity_scope: {city: Bogota, country: CO}
  week_offset: L0W
  tool: get_insight_candidates(entity_scope={city:'Bogota', country:'CO'}, top_n=5)
  [lee insight_candidates.parquet — no re-ejecuta detectores]

TOOL OUTPUT:
  Zona X | persistent_deterioration | Restaurants SS > ATC CVR | 3 sem consecutivas ↓ | HIGH
  Zona Y | anomaly_point            | Gross Profit UE          | robust_z=3.1         | HIGH
  Zona Z | peer_gap                 | Perfect Orders           | -0.12 vs peer median | MEDIUM

BOT:   "Encontré 3 alertas activas en Bogotá (L0W):

        🔴 [HIGH] Zona X — Restaurants SS > ATC CVR lleva 3 semanas consecutivas de caída.
        🔴 [HIGH] Zona Y — Gross Profit UE presenta anomalía puntual (robust_z=3.1).
        🟡 [MEDIUM] Zona Z — Perfect Orders está 0.12 por debajo de su peer group."

CAVEAT: "Thresholds provisionales (wow_delta=10%, decline_streak=3sem, robust_z=2.5).
         Alta tasa de falsos positivos esperada hasta calibración por métrica.
         Direcciones de métricas son provisionales."
VIZ:   alert_cards (una por insight, color=severidad)
NEXT_Q: "¿Ver tendencia de Zona X en CVR? | ¿Filtrar solo alertas HIGH? | ¿Comparar Zona Y con su peer group?"
```

---

#### GF10 — Causalidad fuerte rechazada

```
USER:  Demuéstrame que el CVR causa más órdenes en Chapinero

PLAN:
  intent: hypothesis_request
  → apply_hypothesis_guard() activa
  user asked for 'causa' → forbidden_term detected

BOT:   "No puedo demostrar causalidad con los datos disponibles.
        Lo que sí puedo mostrarte es la asociación observada:
        Restaurants SS > ATC CVR covaría con WoW de órdenes (ρ=+0.229, n=9 semanas).
        Esta es la correlación más alta del dataset, pero no es evidencia de causa-efecto.
        ¿Quiero mostrarte la asociación de forma explícita como hipótesis de trabajo?"

CAVEAT: "No existen condiciones para afirmar causalidad con estos datos:
         9 semanas de historia, correlación observacional, sin experimento controlado."
NEXT_Q: "¿Ver la asociación como hipótesis? | ¿Buscar otros posibles drivers?"
```

---

#### GF07 — Follow-up scope refine

```
[Contexto previo: GF02 — Perfect Orders Wealthy vs Non Wealthy en México]

USER:  ¿Y solo en Colombia?

PLAN:
  intent: follow_up_scope_refine
  → hereda: metric=perfect_orders, comparison=ZONE_TYPE, intent=compare
  → nuevo filtro: country=CO
  tool: compare_segments(metric_id='perfect_orders', segment_a='Wealthy', segment_b='Non Wealthy',
                         filters={'country':'CO'}, week_offset='L0W')

BOT:   "Cambiando el filtro a Colombia (L0W):
        Wealthy: mediana=0.80 (n=12)  |  Non Wealthy: mediana=0.77 (n=21)
        Delta: Wealthy − Non Wealthy = +0.03

        Patrón similar a México aunque el grupo Wealthy es más pequeño en CO."

CAVEAT: "Peer group Wealthy en CO (n=12) está cerca del mínimo recomendado (10).
         Benchmark con cautela."
```

## 11) Evaluacion conversacional

Se proponen criterios medibles para QA de orquestacion y calidad de respuesta.

In [11]:
evaluation_framework = pd.DataFrame([
    {'dimension': 'intent_classification_accuracy', 'definition': 'porcentaje de intents correctamente clasificados', 'target': '>= 0.90'},
    {'dimension': 'parameter_extraction_quality', 'definition': 'extraccion correcta de metric/entity/time/filter', 'target': '>= 0.85'},
    {'dimension': 'execution_correctness', 'definition': 'tool calls correctos vs gold plan', 'target': '>= 0.95'},
    {'dimension': 'faithfulness_groundedness', 'definition': 'respuesta alineada a salida de herramientas', 'target': '>= 0.98'},
    {'dimension': 'clarity', 'definition': 'respuesta entendible y estructurada', 'target': '>= 0.90'},
    {'dimension': 'caveat_quality', 'definition': 'caveats correctos y oportunos', 'target': '>= 0.95'},
    {'dimension': 'follow_up_handling', 'definition': 'mantener contexto sin suposiciones peligrosas', 'target': '>= 0.90'},
])

golden_eval_set = golden_flows[['flow_id', 'user_query', 'intent', 'requires_clarification']].copy()

display(evaluation_framework)
display(golden_eval_set)

,dimension,definition,target
0,intent_classification_accuracy,porcentaje de intents correctamente clasificados,>= 0.90
1,parameter_extraction_quality,extraccion correcta de metric/entity/time/filter,>= 0.85
2,execution_correctness,tool calls correctos vs gold plan,>= 0.95
3,faithfulness_groundedness,respuesta alineada a salida de herramientas,>= 0.98
4,clarity,respuesta entendible y estructurada,>= 0.90
5,caveat_quality,caveats correctos y oportunos,>= 0.95
6,follow_up_handling,mantener contexto sin suposiciones peligrosas,>= 0.90


,flow_id,user_query,intent,requires_clarification
0,GF01,Top 5 zonas con mayor Lead Penetration,rank,True
1,GF02,Compara Perfect Orders entre Wealthy y Non Wea...,compare,False
2,GF03,Evolucion de Gross Profit UE en Chapinero,trend,True
3,GF04,Promedio de Lead Penetration por pais,aggregate,True
4,GF05,Zonas con alto Lead Penetration y bajo Perfect...,multivariable_filter,True
5,GF06,Zonas que mas crecen en orders y que podria ex...,hypothesis_request,False
6,GF07,Y solo en Colombia?,follow_up_scope_refine,False
7,GF08,Muestralo en grafico,follow_up_visualization,False
8,GF09,Que problemas tiene Bogota esta semana?,insight_request,False
9,GF10,Demuestrame que Restaurants SS > ATC CVR causa...,hypothesis_request,True


## 12) Plan de implementacion a app

Conecta este diseño con una app futura (ej. Streamlit) y define dependencias con NB30.

In [12]:
implementation_plan = pd.DataFrame([
    {'phase': 1, 'component': 'planner + intent router', 'depends_on': 'NB20 intents/configs', 'status': 'ready_to_build'},
    {'phase': 2, 'component': 'tool adapter layer', 'depends_on': 'deterministic functions + NB30 outputs', 'status': 'ready_to_build'},
    {'phase': 3, 'component': 'response renderer', 'depends_on': 'response_contract + language rules', 'status': 'ready_to_build'},
    {'phase': 4, 'component': 'chat state manager', 'depends_on': 'chat_state_schema', 'status': 'ready_to_build'},
    {'phase': 5, 'component': 'golden flow tests', 'depends_on': 'golden_flows + evaluation framework', 'status': 'ready_to_build'},
    {'phase': 6, 'component': 'streamlit app shell', 'depends_on': 'phases 1-5', 'status': 'pending_implementation'},
])

display(implementation_plan)

,phase,component,depends_on,status
0,1,planner + intent router,NB20 intents/configs,ready_to_build
1,2,tool adapter layer,deterministic functions + NB30 outputs,ready_to_build
2,3,response renderer,response_contract + language rules,ready_to_build
3,4,chat state manager,chat_state_schema,ready_to_build
4,5,golden flow tests,golden_flows + evaluation framework,ready_to_build
5,6,streamlit app shell,phases 1-5,pending_implementation


## 13) Exportes de diseño

Este bloque materializa la documentación del contrato conversacional para implementación y auditoría.

In [ ]:
def markdown_table(df: 'pd.DataFrame') -> str:
    if df.empty: return '(sin filas)'
    cols = [str(c) for c in df.columns]
    lines = ['| ' + ' | '.join(cols) + ' |', '| ' + ' | '.join(['---'] * len(cols)) + ' |']
    for _, row in df.iterrows():
        vals = [str(row[c]).replace('\n',' ')[:120] for c in df.columns]
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines)

now_ts = datetime.now(timezone.utc).isoformat()

# ── 1. docs/architecture/reto1_chatbot_contract.md ────────────────────────
contract_sections = [
    f'# Reto 1 — Chatbot Conversational Contract',
    f'> Generated: {now_ts}',
    '',
    '## Architectural Principles',
    '',
    '| Principle | Rule |',
    '|---|---|',
    '| LLM role | Planner + orchestrator + controlled renderer. Never computes analytics. |',
    '| Tool role | All calculations and retrievals are deterministic functions (NB30 outputs or semantic layer). |',
    '| Safety | No causal claims. No improvised analytics. No suspended/invalid metrics used silently. |',
    '| Transparency | Every response includes evidence scope, caveats, and next-question suggestions. |',
    '',
    '## System Flow',
    '',
    '```',
    'User input → LLM Planner → validate_plan() → tool calls → LLM Renderer → structured response',
    '                                                                           (language guards applied)',
    '```',
    '',
    '## Intent Operational Map',
    '',
    markdown_table(intent_view),
    '',
    '## Planner Schema (ChatbotExecutionPlan)',
    '',
    '```json',
    json.dumps(planner_schema, indent=2, ensure_ascii=False),
    '```',
    '',
    '## Chat Session State (ChatSessionState)',
    '',
    '```json',
    json.dumps(chat_state_schema, indent=2, ensure_ascii=False),
    '```',
    '',
    '## Clarification Rules',
    '',
    markdown_table(clarification_rules),
    '',
    '## Tool Contract',
    '',
    markdown_table(tool_contract),
    '',
    '## Response Contract',
    '',
    markdown_table(response_contract),
    '',
    '## Language Rules',
    '',
    markdown_table(language_ops),
    '',
    '## UX Components by Intent',
    '',
    markdown_table(ux_components),
    '',
    '## Implementation Plan',
    '',
    markdown_table(implementation_plan),
    '',
    '## Open Items',
    '',
    '- [ ] Calibrate detector thresholds (NB30) per metric before using in production',
    '- [ ] Validate desired_direction of 13 metrics with business team',
    '- [ ] Implement planner as actual LLM structured-output prompt',
    '- [ ] Build tool adapter layer (stubs already defined in tool_contract)',
    '- [ ] Wire golden flows as automated tests',
    '- [ ] Build Streamlit shell consuming this contract',
]
contract_text = '\n'.join(contract_sections)
(ARCH_DIR / 'reto1_chatbot_contract.md').write_text(contract_text, encoding='utf-8')
print(f'contract  → {ARCH_DIR / "reto1_chatbot_contract.md"} ({len(contract_text)} chars)')

# ── 2. reports/reto1/chatbot_design_summary.md ────────────────────────────
summary_sections = [
    f'# Chatbot Design Summary — Reto 1',
    f'> Generated: {now_ts}',
    '',
    '## What the chatbot does',
    '- Classifies user intent against question_types.yaml (NB20)',
    '- Extracts parameters: metric, entity, time_window, filters, comparison',
    '- Validates plan against semantic contract (validate_plan())',
    '- Routes to deterministic tools or NB30 insight_candidates',
    '- Renders response using response_contract + language guards',
    '- Manages minimal conversation state for follow-ups',
    '',
    '## What the chatbot does NOT do',
    '- Does not compute metrics from raw data',
    '- Does not make causal claims',
    '- Does not silently use suspended metrics (lead_penetration)',
    '- Does not compare invalid segment pairs (per NB20 not_comparable)',
    '- Does not improvise when out of scope',
    '',
    '## Core design decisions',
    '',
    '| Decision | Rationale |',
    '|---|---|',
    '| Planner as structured JSON | Gated execution: invalid plans cannot reach tools |',
    '| validate_plan() pre-flight | Semantic contract enforcement before any tool call |',
    '| Response contract | Consistent format: short_answer + evidence + caveat + next_q |',
    '| Language guards | Non-causal framing, provisional metric warnings, peer group size warnings |',
    '| Minimal chat state | Only last metric/entity/filter/mode — no unbounded memory |',
    '| NB30 pre-computed outputs | Chatbot reads insight_candidates.parquet — does not re-run detectors |',
    '',
    '## Intent support levels',
    '',
    markdown_table(intent_view[['intent_id','display_name','support_level']]),
    '',
    '## Implementation readiness',
    '',
    markdown_table(implementation_plan),
    '',
    '## What remains before production',
    '',
    '1. **Calibrate thresholds** in config/business_rules.yaml per metric',
    '2. **Validate desired_direction** of all 13 metrics with business team',
    '3. **Implement planner prompt** (LLM structured output for ChatbotExecutionPlan)',
    '4. **Build tool adapter layer** (wrap NB30 functions + data lookups)',
    '5. **Wire golden flows** as automated regression tests',
    '6. **Build Streamlit app** consuming contract defined here',
]
summary_text = '\n'.join(summary_sections)
(REPORTS_DIR / 'chatbot_design_summary.md').write_text(summary_text, encoding='utf-8')
print(f'summary   → {REPORTS_DIR / "chatbot_design_summary.md"} ({len(summary_text)} chars)')

# ── 3. reports/reto1/golden_conversation_flows.md ────────────────────────
flows_sections = [
    f'# Golden Conversation Flows — Reto 1',
    f'> Generated: {now_ts}',
    '',
    '## Flow Summary',
    '',
    markdown_table(golden_flows),
    '',
    '## Detail per flow',
    '',
    '### GF01 — Suspended metric (rank)',
    '- **User**: "Top 5 zonas con mayor Lead Penetration"',
    '- **Plan**: intent=rank, metric=lead_penetration → REJECTED (suspended_pending_definition)',
    '- **Response**: rechazar, explicar suspensión, proponer alternativa (perfect_orders, pro_adoption)',
    '- **Caveat**: metric_suspended — max observado 393.9, denominador no clarificado',
    '',
    '### GF02 — Compare segments (valid)',
    '- **User**: "Compara Perfect Orders entre Wealthy y Non Wealthy en México"',
    '- **Plan**: intent=compare, metric=perfect_orders, entity={country:MX, segment:ZONE_TYPE}',
    '- **Tool**: compare_segments → Wealthy=0.834(n=34) vs Non Wealthy=0.791(n=45), delta=+0.043',
    '- **Response**: tabla comparativa con delta y contexto de tamaño de grupos',
    '- **Caveat**: dirección provisional; definición de "perfecto" puede variar entre mercados',
    '',
    '### GF03 — Trend con zona ambigua',
    '- **User**: "Evolución de Gross Profit UE en Chapinero"',
    '- **Plan**: intent=trend, metric=gross_profit_ue, zone=Chapinero → CLARIFY (zona ambigua)',
    '- **Clarification**: "¿Chapinero en Bogotá, Colombia?"',
    '- **Tool**: get_trend(CO|Bogota|Chapinero, L8W→L0W) → serie con recuperación L5W→L1W',
    '- **Caveat**: outlier_risk=high; 9 semanas insuficientes para tendencia robusta',
    '',
    '### GF06 — Hypothesis non-causal',
    '- **User**: "¿Qué podría explicar el crecimiento de órdenes en zonas con alto CVR?"',
    '- **Plan**: intent=hypothesis_request, metric=orders_total, tool=generate_hypothesis_candidates',
    '- **Tool**: Spearman WoW → restaurants_ss_atc_cvr ρ=+0.229 (highest in dataset)',
    '- **Response**: posibles drivers con framing explícitamente no causal',
    '- **Caveat**: correlaciones WoW n=9 semanas — evidencia débil, no para tomar decisiones',
    '',
    '### GF07 — Follow-up scope refine',
    '- **User**: [after GF02] "¿Y solo en Colombia?"',
    '- **Plan**: intent=follow_up_scope_refine → hereda metric+intent previos, agrega country=CO',
    '- **Tool**: compare_segments → Wealthy=0.80(n=12) vs Non Wealthy=0.77(n=21)',
    '- **Caveat**: Wealthy en CO (n=12) cerca del mínimo recomendado — benchmark con cautela',
    '',
    '### GF08 — Follow-up visualization',
    '- **User**: [after any table result] "Muéstralo en gráfico"',
    '- **Plan**: intent=follow_up_visualization → tool=render_chart_data(last_result)',
    '- **Response**: misma data, transformada a chart spec (no recalcula)',
    '',
    '### GF09 — Insight request (NB30 routing)',
    '- **User**: "¿Qué problemas tiene Bogotá esta semana?"',
    '- **Plan**: intent=insight_request, entity={city:Bogota, country:CO}',
    '- **Tool**: route_insight_request → lee insight_candidates.parquet (no re-ejecuta detectors)',
    '- **Response**: alert_cards por zona (persistent_deterioration, anomaly_point, peer_gap)',
    '- **Caveat**: thresholds provisionales — alta tasa de falsos positivos hasta calibración',
    '',
    '### GF10 — Causalidad fuerte rechazada',
    '- **User**: "Demuéstrame que el CVR causa más órdenes en Chapinero"',
    '- **Plan**: intent=hypothesis_request → apply_hypothesis_guard detects "causa" (forbidden term)',
    '- **Response**: rechazar causalidad, ofrecer evidencia asociativa como alternativa',
    '- **Caveat**: no existe condición para afirmar causalidad con 9 semanas de datos observacionales',
    '',
    '## Evaluation targets',
    '',
    '| Dimension | Target |',
    '|---|---|',
    '| Intent classification accuracy | >= 0.90 |',
    '| Parameter extraction quality | >= 0.85 |',
    '| Execution correctness | >= 0.95 |',
    '| Faithfulness / groundedness | >= 0.98 |',
    '| Caveat quality | >= 0.95 |',
    '| Follow-up handling | >= 0.85 |',
]
flows_text = '\n'.join(flows_sections)
(REPORTS_DIR / 'golden_conversation_flows.md').write_text(flows_text, encoding='utf-8')
print(f'flows     → {REPORTS_DIR / "golden_conversation_flows.md"} ({len(flows_text)} chars)')

# ── 4. reports/reto1/chatbot_planner_schema.json ──────────────────────────
planner_full = {
    'schema': planner_schema,
    'chat_state_schema': chat_state_schema,
    'tool_contract': tool_contract.to_dict(orient='records'),
    'response_contract': response_contract.to_dict(orient='records'),
    'generated_at': now_ts,
}
(REPORTS_DIR / 'chatbot_planner_schema.json').write_text(
    json.dumps(planner_full, indent=2, ensure_ascii=False, default=str),
    encoding='utf-8',
)
print(f'schema    → {REPORTS_DIR / "chatbot_planner_schema.json"}')
print()
print('=== All outputs generated ===')